In [0]:
from pyspark.sql import functions as F, Window

def clean_customers(df):
    """Trim keys, clean names, drop null/duplicate keys."""
    return (
        df.withColumn("customer_id", F.trim("customer_id"))
          .dropna(subset=["customer_id"])
          .withColumn("customer_name", F.initcap(F.trim(F.regexp_replace("customer_name", r"[^A-Za-z\s\.\-']", ""))))
          .withColumn("customer_name", F.regexp_replace("customer_name", r"\s+", " "))
          .withColumn("customer_name", F.when(F.col("customer_name") == "", None).otherwise(F.col("customer_name")))
          .dropDuplicates(["customer_id"])
          .select("customer_id", "customer_name", "email", "phone", "segment", "country", "city", "state", "postal_code", "region")
    )

def clean_products(df):
    """Trim keys, cast price, keep ONE row per product_id."""
    w = Window.partitionBy("product_id").orderBy(F.col("state"), F.col("product_name"))
    return (
        df.withColumn("product_id", F.trim("product_id"))
          .dropna(subset=["product_id"])
          .withColumn("price_per_product", F.col("price_per_product").cast("double"))
          .withColumn("_rn", F.row_number().over(w))
          .filter("_rn = 1")
          .drop("_rn")
          .select("product_id", "category", "sub_category", "product_name", "state", "price_per_product")
    )

def build_enriched_orders(orders, customers, products):
    """Order info + customer name/country + product category/sub-category.
    Left joins keep every order. Profit is rounded to 2 decimals. Adds `year`.
    """
    o = (
        orders
        .withColumn("order_date", F.to_date("order_date", "d/M/yyyy"))
        .withColumn("ship_date", F.to_date("ship_date", "d/M/yyyy"))
        .withColumn("profit", F.round(F.col("profit"), 2))
        .withColumn("year", F.year("order_date"))
    )
    c = customers.select("customer_id", "customer_name", "country")
    p = products.select("product_id", "category", "sub_category")
    return (
        o.join(F.broadcast(c), "customer_id", "left")
         .join(F.broadcast(p), "product_id", "left")
         .select("row_id", "order_id", "order_date", "ship_date", "year", "ship_mode",
                 "customer_id", "customer_name", "country",
                 "product_id", "category", "sub_category",
                 "quantity", "price", "discount", "profit")
    )

def build_profit_aggregate(enriched_orders):
    """Total profit by year, category, sub-category and customer."""
    return enriched_orders.groupBy("year", "category", "sub_category", "customer_id", "customer_name").agg(F.round(F.sum("profit"), 2).alias("total_profit"))


def save_table(df, table, expected_min_rows=1, overwrite_schema=False):
    """Write df as a Delta table, failing loudly on empty or unexpected output."""
    try:
        n = df.count()
        if n < expected_min_rows:
            raise ValueError(f"{table}: only {n} rows, expected at least {expected_min_rows}")
        writer = df.write.format("delta").mode("overwrite")
        if overwrite_schema:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(table)
        print(f"{table}: wrote {n} rows")
    except Exception as e:
        print(f"FAILED writing {table}: {e}")
        raise